In [3]:
import duckdb as db

In [5]:
from pathlib import Path
import json
import numpy as np

arrays = sorted(Path("outputs").glob("*.json"))


In [41]:
len(arrays)

10000

In [6]:
connection = db.connect("publications_princeton.36608975.db")

connection.sql("SHOW TABLES")

connection.sql("SELECT * FROM figure_property")

connection.sql("DESCRIBE figure")

paths = connection.sql("SELECT server_path FROM figure")

from pathlib import Path

figure_id_ordered_paths = [f"array_fig-{Path(pth).parent.name}-{Path(pth).stem}.json" for pth in paths.df().server_path]

figure_id_ordered_paths[0]

exists_check = [Path(f"outputs/{p}").exists() for p in figure_id_ordered_paths]

# check to see how many figures in the table weren't chart typed
sum([not Path(f"outputs/{p}").exists() for p in figure_id_ordered_paths])

# now create an array that maps from the order in the figure_id_ordered_paths to the indices of the original json arrays list which we built the arrays fr
lookup_map = {str(e.name):i for i,e in enumerate(arrays)}

arrays[:5]

exists_check[:5]

# example for getting the index in our result array for the figure who's table id is 0
ex_figure_id_0 =figure_id_ordered_paths[0]
print(ex_figure_id_0)
array_index = lookup_map[ex_figure_id_0]
print(array_index)
# then we can use that same index to pull out the x,y,z coordinates that are supposed to replace the existing ones

array_fig-4281966491-Figure1-1.json
6217


In [52]:
connection.sql("SELECT id,server_path FROM figure")

┌───────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│  id   │                                                 server_path                                                  │
│ int64 │                                                   varchar                                                    │
├───────┼──────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│     0 │ https://data.cyverse.org/dav-anon/iplant/home/carolinarr/vis-sieve/Princeton_content/4281966491/Figure1-1.…  │
│     3 │ https://data.cyverse.org/dav-anon/iplant/home/carolinarr/vis-sieve/Princeton_content/4210714446/Figure10-1…  │
│     6 │ https://data.cyverse.org/dav-anon/iplant/home/carolinarr/vis-sieve/Princeton_content/4210714446/Figure6-1.…  │
│     9 │ https://data.cyverse.org/dav-anon/iplant/home/carolinarr/vis-sieve/Princeton_content/4210714446/Figure1-1.…  │
│    12 │ https://data.cyverse.o

In [47]:
lookup_map

{'array_fig-2340671844-Figure1-1.json': 0,
 'array_fig-2340671844-Figure10-1.json': 1,
 'array_fig-2340671844-Figure11-1.json': 2,
 'array_fig-2340671844-Figure12-1.json': 3,
 'array_fig-2340671844-Figure13-1.json': 4,
 'array_fig-2340671844-Figure14-1.json': 5,
 'array_fig-2340671844-Figure15-1.json': 6,
 'array_fig-2340671844-Figure16-1.json': 7,
 'array_fig-2340671844-Figure17-1.json': 8,
 'array_fig-2340671844-Figure18-1.json': 9,
 'array_fig-2340671844-Figure19-1.json': 10,
 'array_fig-2340671844-Figure2-1.json': 11,
 'array_fig-2340671844-Figure20-1.json': 12,
 'array_fig-2340671844-Figure21-1.json': 13,
 'array_fig-2340671844-Figure22-1.json': 14,
 'array_fig-2340671844-Figure23-1.json': 15,
 'array_fig-2340671844-Figure24-1.json': 16,
 'array_fig-2340671844-Figure25-1.json': 17,
 'array_fig-2340671844-Figure26-1.json': 18,
 'array_fig-2340671844-Figure27-1.json': 19,
 'array_fig-2340671844-Figure28-1.json': 20,
 'array_fig-2340671844-Figure29-1.json': 21,
 'array_fig-2340671844

In [42]:
# mystery
# why are there tons of figures listed multiple times
counter = {}
for f in figure_id_ordered_paths:
    count = counter.get(f,0)
    count+=1
    counter[f] =count
    
counter

{'array_fig-4281966491-Figure1-1.json': 2,
 'array_fig-4210714446-Figure10-1.json': 3,
 'array_fig-4210714446-Figure6-1.json': 3,
 'array_fig-4210714446-Figure1-1.json': 3,
 'array_fig-4210714446-Figure4-1.json': 3,
 'array_fig-4210714446-Figure2-1.json': 3,
 'array_fig-4210714446-Figure9-1.json': 3,
 'array_fig-4210714446-Figure7-1.json': 3,
 'array_fig-4210714446-Figure3-1.json': 3,
 'array_fig-4210714446-Figure8-1.json': 3,
 'array_fig-4210712567-Figure1-1.json': 3,
 'array_fig-4210712567-Figure4-1.json': 3,
 'array_fig-4210712567-Figure2-1.json': 3,
 'array_fig-4210712567-Figure5-1.json': 3,
 'array_fig-4210712567-Figure3-1.json': 3,
 'array_fig-4292117664-Figure1-1.json': 2,
 'array_fig-4292117664-Figure4-1.json': 2,
 'array_fig-4292117664-Figure2-1.json': 2,
 'array_fig-4226179712-Figure6-1.json': 2,
 'array_fig-4226179712-Figure1-1.json': 2,
 'array_fig-2340671844-Figure1-1.json': 2,
 'array_fig-2340671844-Figure10-1.json': 2,
 'array_fig-2340671844-Figure11-1.json': 2,
 'array_

In [8]:
embedding = np.load("./embedded_chart_types_15_2d.npy")

In [9]:
fig_prop_df = connection.sql("SELECT * FROM figure_property").df()

fig_prop_df


# this removes any of the missing figures so that we update just the correct ones
fig_prop_df[exists_check]

# entirely possible that the updates are going to be easier as pandas instead of duckdb 

# demonstration of reordering based on indexing

rand_test = (np.random.random((15,3))*10).astype("uint8")

rand_test

inds = np.random.choice(np.arange(15),15,replace=False)

inds

rand_test[inds]

array_indices = [lookup_map[fig_path] for i,fig_path in enumerate(figure_id_ordered_paths) if exists_check[i] ]
array_indices

table_ordered_embeddings = embedding[array_indices]
table_ordered_embeddings

array([[ 33.39754 ,  20.890371],
       [ 52.282734, -60.680984],
       [ 34.247505,  56.440586],
       ...,
       [ 19.96133 , -46.644093],
       [ 96.64834 , -46.418762],
       [ 68.81347 ,  10.035036]], dtype=float32)

In [53]:
array_indices

array([6217, 2877, 2881, ..., 9997, 9998, 9999])

In [54]:
arrays[array_indices[0]]

PosixPath('outputs/array_fig-4281966491-Figure1-1.json')

In [38]:
table_ordered_embeddings.shape

(12911, 2)

In [37]:
embedding.shape

(10000, 2)

In [35]:
array_indices.shape

(12911,)

In [33]:
table_ordered_embeddings.shape

(12911, 2)

In [39]:
len(exists_check)

14486

In [40]:
sum(exists_check)

12911

In [10]:
array_indices

[6217,
 2877,
 2881,
 2876,
 2880,
 2878,
 2884,
 2882,
 2879,
 2883,
 2871,
 2874,
 2872,
 2875,
 2873,
 7776,
 7778,
 7777,
 5350,
 5345,
 0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161

In [43]:
np.column_stack([np.array(array_indices)[:,None],table_ordered_embeddings])

array([[6217.        ,   33.39754105,   20.89037132],
       [2877.        ,   52.28273392,  -60.6809845 ],
       [2881.        ,   34.24750519,   56.44058609],
       ...,
       [9997.        ,   19.96133041,  -46.64409256],
       [9998.        ,   96.64833832,  -46.41876221],
       [9999.        ,   68.81346893,   10.03503609]])

In [18]:
import pandas as pd

In [61]:
og_df  = connection.sql("SELECT * FROM figure_property").df()

In [62]:
og_df

,figure_id,name,int_value,string_value,xPos,yPos,zPos,score
0,0,name-1,1,uigXfXok,0.188985,13.698142,-10.303946,2760
1,1,name-42,42,LrgopzrG,8.893129,8.880141,9.741223,81195
2,2,name-4,4,MXeDVNQU,6.284884,10.211571,4.502829,77555
3,3,name-6,6,hYUFJQMX,7.729008,4.585903,-9.169049,85731
4,4,name-3,3,oNTzMXQF,25.651455,9.985393,-1.703986,47666
...,...,...,...,...,...,...,...,...
14481,14481,name-14,14,kaAvZsLf,-1105.000000,87.000000,-390.000000,58555
14482,14482,name-41,41,zxjZmkcU,-895.000000,65.000000,695.000000,43870
14483,14483,name-19,19,qAkCGvny,720.000000,3.000000,-240.000000,2382
14484,14484,name-19,19,EVAYXLry,540.000000,87.000000,-375.000000,58663


In [64]:
len(exists_check)

14486

In [63]:
og_df.loc[exists_check,]

,figure_id,name,int_value,string_value,xPos,yPos,zPos,score
0,0,name-1,1,uigXfXok,0.188985,13.698142,-10.303946,2760
1,1,name-42,42,LrgopzrG,8.893129,8.880141,9.741223,81195
2,2,name-4,4,MXeDVNQU,6.284884,10.211571,4.502829,77555
3,3,name-6,6,hYUFJQMX,7.729008,4.585903,-9.169049,85731
4,4,name-3,3,oNTzMXQF,25.651455,9.985393,-1.703986,47666
...,...,...,...,...,...,...,...,...
12906,12906,name-13,13,XxgvlxIs,-7.725756,6.583314,10.997828,30096
12907,12907,name-26,26,hzlXrbLS,7.837799,5.933046,11.120034,16127
12908,12908,name-22,22,DtEsuSar,-14.693877,-9.589152,4.078270,52773
12909,12909,name-7,7,lnamOXhQ,18.914112,-10.648349,-5.735243,75086


In [ ]:
duckdb.sql("CREATE TABLE 2d_tsne AS SELECT * FROM my_df")

In [65]:
duckdb.close()

In [66]:
connection = db.connect("test.db")

In [67]:
connection.sql("SHOW TABLES")

┌─────────────────┐
│      name       │
│     varchar     │
├─────────────────┤
│ author          │
│ charts          │
│ contribution    │
│ figure          │
│ figure_property │
│ institution     │
│ keyword         │
│ paper           │
│ residence       │
│ topic           │
│ work_keyword    │
│ work_topic      │
├─────────────────┤
│     12 rows     │
└─────────────────┘

In [68]:
table_ordered_embeddings

array([[ 33.39754 ,  20.890371],
       [ 52.282734, -60.680984],
       [ 34.247505,  56.440586],
       ...,
       [ 19.96133 , -46.644093],
       [ 96.64834 , -46.418762],
       [ 68.81347 ,  10.035036]], dtype=float32)

In [72]:
connection.sql("SELECT * FROM figure_property")

┌───────────┬─────────┬───────────┬──────────────┬───────┬───────┬───────┬───────┐
│ figure_id │  name   │ int_value │ string_value │ xPos  │ yPos  │ zPos  │ score │
│   int64   │ varchar │   int32   │   varchar    │ int32 │ int32 │ int32 │ int32 │
├───────────┼─────────┼───────────┼──────────────┼───────┼───────┼───────┼───────┤
│         0 │ name-1  │         1 │ uigXfXok     │     0 │     0 │  -895 │  2760 │
│         1 │ name-42 │        42 │ LrgopzrG     │     0 │     0 │   860 │ 81195 │
│         2 │ name-4  │         4 │ MXeDVNQU     │     0 │     0 │ -1075 │ 77555 │
│         3 │ name-6  │         6 │ hYUFJQMX     │     0 │     0 │ -1075 │ 85731 │
│         4 │ name-3  │         3 │ oNTzMXQF     │     0 │     0 │ -1000 │ 47666 │
│         5 │ name-24 │        24 │ FHBvveQu     │     0 │     0 │    65 │  7111 │
│         6 │ name-24 │        24 │ ZHmcbpIU     │     0 │     0 │     5 │ 31565 │
│         7 │ name-30 │        30 │ sPmAjulQ     │     0 │     0 │   190 │ 87449 │
│   

In [71]:
for i,row in enumerate(table_ordered_embeddings):
    x,y = row
    connection.sql(f"UPDATE figure_property SET xPos = {0} WHERE figure_id = {i} ")
    connection.sql(f"UPDATE figure_property SET yPos = {0} WHERE figure_id = {i} ")

In [94]:
connection.close()

In [78]:
connection = db.connect("test.db")

In [79]:
connection.sql("CHECKPOINT")

FatalException: FATAL Error: Failed to create checkpoint because of error: INTERNAL Error: Unsupported compression function type

In [117]:
connection = db.connect("test.db")

In [118]:
connection.sql("SHOW TABLES").df()

,name
0,author
1,charts
2,contribution
3,figure
4,figure_property
5,institution
6,keyword
7,paper
8,residence
9,topic


In [119]:
for tbl in connection.sql("SHOW TABLES").df().name:
    if tbl == "charts" or tbl == "work_keyword" or tbl == "work_topic":
        continue
    print(tbl)
    connection.sql(f"COPY {tbl} TO '{tbl}.csv' (HEADER, DELIMITER ',');")

author
contribution
figure
figure_property
institution
keyword
paper
residence
topic


In [123]:
new_con = db.connect("new.db")

In [128]:
for tbl in connection.sql("SHOW TABLES").df().name:
    if tbl == "charts" or tbl == "work_keyword" or tbl == "work_topic":
        continue
    print(tbl)
    new_con.sql(f"CREATE TABLE {tbl} AS SELECT * FROM read_csv('{tbl}.csv', AUTO_DETECT=TRUE);")

author
contribution
figure
figure_property
institution
keyword
paper
residence
topic


In [129]:
new_con.sql("SHOW TABLES")

┌─────────────────┐
│      name       │
│     varchar     │
├─────────────────┤
│ author          │
│ contribution    │
│ figure          │
│ figure_property │
│ institution     │
│ keyword         │
│ paper           │
│ residence       │
│ topic           │
└─────────────────┘

In [130]:
[print(new_con.sql(f"DESCRIBE TABLE {tbl}")) for tbl in new_con.sql("SHOW TABLES").df().name]

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ name        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ au_id       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ paper_id    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

┌─────────────┬───

[None, None, None, None, None, None, None, None, None]

In [122]:
connection.sql("DESCRIBE TABLE work_keyword")

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ work_id     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_id  │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ score       │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [121]:
connection.sql("DESCRIBE TABLE charts")

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id          │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ chart       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ xPos        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ yPos        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ zPos        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [120]:
connection.sql("DESCRIBE TABLE work_topic")

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ work_id     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ topic_id    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ score       │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [116]:
connection.close()

In [89]:
!wget "https://zhiyangwang.site/vis-sieve/_file/data/publications_princeton.36608975.db"

/bin/bash: wget: command not found


In [131]:
table_ordered_embeddings

array([[ 33.39754 ,  20.890371],
       [ 52.282734, -60.680984],
       [ 34.247505,  56.440586],
       ...,
       [ 19.96133 , -46.644093],
       [ 96.64834 , -46.418762],
       [ 68.81347 ,  10.035036]], dtype=float32)

In [133]:
og_df.describe()

,figure_id,int_value,xPos,yPos,zPos,score
count,14486.000000,14486.000000,14486.000000,14486.000000,14486.000000,14486.000000
mean,7242.500000,22.022643,-2.311886,8.004975,-12.791814,50105.899213
std,4181.892335,12.876033,206.565075,29.192617,201.710815,29007.057800
min,0.000000,0.000000,-1120.000000,-26.448812,-1120.000000,12.000000
25%,3621.250000,11.000000,-9.987364,-6.293885,-9.056345,25115.500000
50%,7242.500000,22.000000,-0.982851,1.461141,-0.044043,50066.500000
75%,10863.750000,33.000000,9.178718,9.904573,8.333774,75576.250000
max,14485.000000,44.000000,1095.000000,150.000000,1055.000000,99996.000000


In [134]:
len(array_indices)

12911

In [135]:
og_df[:len(array_indices)].describe()

,figure_id,int_value,xPos,yPos,zPos,score
count,12911.000000,12911.000000,12911.000000,12911.000000,12911.000000,12911.000000
mean,6455.000000,22.014639,0.199444,-0.217329,-0.334383,50045.166292
std,3727.228998,12.886414,11.682908,9.832524,10.179600,28976.881180
min,0.000000,0.000000,-22.817026,-26.448812,-26.517918,12.000000
25%,3227.500000,11.000000,-8.510283,-7.435060,-7.735101,25087.000000
50%,6455.000000,22.000000,-0.915122,-0.010774,0.180804,50039.000000
75%,9682.500000,33.000000,8.016684,7.370668,7.470092,75476.000000
max,12910.000000,44.000000,29.542242,22.441153,24.401154,99996.000000


In [136]:
new_con.close()